In [9]:
import os
os.environ["HF_HOME"] = "D:/huggingface_cache"

In [10]:
#Paths
RESUMES_DIR   = "../data/raw/resumes"          # subfolders = categories
IMAGES_DIR    = "../data/raw/page_images"      # output: one PNG per page
DOCTAGS_FILE  = "../data/extracted_text/doctags_progress.txt"

os.makedirs(IMAGES_DIR, exist_ok=True)
os.makedirs(os.path.dirname(DOCTAGS_FILE), exist_ok=True)
print("Paths configured.")

Paths configured.


In [11]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_NAME = "ibm-granite/granite-docling-258M"
device     = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

granite_doc_processor = AutoProcessor.from_pretrained(MODEL_NAME)
granite_doc_model     = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,
).to(device)

granite_doc_model.eval()
print(f"Granite Docling loaded on {device}")

Using device: cuda
Granite Docling loaded on cuda


In [12]:
import pymupdf
from PIL import Image
import io
import json
from pathlib import Path

def pdf_to_images(
    resumes_dir: str,
    output_dir: str,
    dpi: int = 200,              # 200 DPI = good quality for resumes without excessive file size
    skip_existing: bool = True,  # skip pages already converted (safe to re-run)
):
    """
    Returns: (image_paths: list[str], metadata: list[dict])
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    meta_path = output_dir / "metadata.json"

    # Load existing metadata so we can skip already-processed files
    if skip_existing and meta_path.exists():
        with open(meta_path, encoding="utf-8") as f:
            existing_meta = json.load(f)
        existing_images = {e["image_path"] for e in existing_meta}
        print(f"Resuming: {len(existing_meta)} pages already processed")
    else:
        existing_meta   = []
        existing_images = set()

    # Discover all PDFs, sorted by (category, filename) for reproducibility
    resumes_dir = Path(resumes_dir)
    pdf_files = sorted(resumes_dir.rglob("*.pdf")) + sorted(resumes_dir.rglob("*.PDF"))

    if not pdf_files:
        print(f"No PDFs found under {resumes_dir}")
        return [], []

    print(f"Found {len(pdf_files)} PDFs across all category folders")

    new_meta = []

    for pdf_path in pdf_files:
        # Category = immediate parent folder name (e.g. "data science resumes")
        category = pdf_path.parent.name
        pdf_stem = pdf_path.stem   # filename without extension

        try:
            doc = pymupdf.open(str(pdf_path))
        except Exception as e:
            print(f"  SKIP (cannot open): {pdf_path.name} — {e}")
            continue

        num_pages = len(doc)

        for page_num, page in enumerate(doc):
            # Use category + stem + page for unique, collision-free filename
            safe_cat  = category.replace(" ", "_").replace("/", "-")
            safe_stem = pdf_stem.replace(" ", "_")[:60]  # cap length
            img_name  = f"{safe_cat}__{safe_stem}__p{page_num:04d}.png"
            img_path  = str(output_dir / img_name)

            if skip_existing and img_path in existing_images:
                continue

            pix         = page.get_pixmap(dpi=dpi)
            image_bytes = pix.tobytes("png")
            image       = Image.open(io.BytesIO(image_bytes))
            image.save(img_path)
            del image

            new_meta.append({
                "image_path"  : img_path,
                "pdf_path"    : str(pdf_path),
                "pdf_name"    : pdf_path.name,
                "pdf_stem"    : pdf_stem,
                "category"    : category,         # ← evaluation label
                "page_number" : page_num,
                "total_pages" : num_pages,
            })

        doc.close()
        if num_pages > 0:
            print(f"  {category}/{pdf_path.name}: {num_pages} pages")

    # Merge with existing and save
    all_meta = existing_meta + new_meta
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(all_meta, f, indent=2)

    image_paths = [e["image_path"] for e in all_meta]

    print(f"\nTotal pages: {len(image_paths)} ({len(new_meta)} new this run)")
    print(f"Metadata: {meta_path}")
    return image_paths, all_meta




In [13]:
# Run
image_paths, metadata = pdf_to_images(RESUMES_DIR, IMAGES_DIR)
print(f"{len(image_paths)} total page images ready")

Resuming: 5172 pages already processed
Found 5172 PDFs across all category folders
  Accountant resumes/Image_10.pdf: 1 pages
  Accountant resumes/Image_11.pdf: 1 pages
  Accountant resumes/Image_12.pdf: 1 pages
  Accountant resumes/Image_14.pdf: 1 pages
  Accountant resumes/Image_16.pdf: 1 pages
  Accountant resumes/Image_17.pdf: 1 pages
  Accountant resumes/Image_18.pdf: 1 pages
  Accountant resumes/Image_19.pdf: 1 pages
  Accountant resumes/Image_2.pdf: 1 pages
  Accountant resumes/Image_20.pdf: 1 pages
  Accountant resumes/Image_21.pdf: 1 pages
  Accountant resumes/Image_22.pdf: 1 pages
  Accountant resumes/Image_24.pdf: 1 pages
  Accountant resumes/Image_25.pdf: 1 pages
  Accountant resumes/Image_26.pdf: 1 pages
  Accountant resumes/Image_29.pdf: 1 pages
  Accountant resumes/Image_30.pdf: 1 pages
  Accountant resumes/Image_31.pdf: 1 pages
  Accountant resumes/Image_32.pdf: 1 pages
  Accountant resumes/Image_33.pdf: 1 pages
  Accountant resumes/Image_34.pdf: 1 pages
  Accountant re

In [14]:
#Sanity Check
import json
from pathlib import Path
from collections import Counter

meta_path = Path(IMAGES_DIR) / "metadata.json"
with open(meta_path, encoding="utf-8") as f:
    metadata = json.load(f)

image_paths = [e["image_path"] for e in metadata]

print(f"Total pages: {len(image_paths)}")
print(f"\nPages per category:")
cat_counts = Counter(e["category"] for e in metadata)
for cat, count in sorted(cat_counts.items()):
    print(f"  {cat:45s} {count:>4} pages")

print(f"\nSample metadata entries:")
for e in metadata[:3]:
    print(f"  {e}")

Total pages: 5172

Pages per category:
  Accountant resumes                             134 pages
  Advocate resumes                               188 pages
  Agricultural resumes                           136 pages
  Apparel resumes                                102 pages
  Architects resumes                             120 pages
  Arts resumes                                   136 pages
  Automobile resumes                              80 pages
  Aviation resumes                               102 pages
  BPO resumes                                     58 pages
  Banking resumes                                 82 pages
  Blockchain resumes                              16 pages
  Building _Construction resumes                 110 pages
  Business Analyst resumes                       112 pages
  Civil Engineer resumes                         146 pages
  Consultant resumes                             142 pages
  Database resumes                               126 pages
  Designing resum

In [15]:
#Granite Prompt
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "Convert this page to docling."},
        ],
    },
]

prompt = granite_doc_processor.apply_chat_template(
    conversation=messages,
    add_generation_prompt=True,
)
print("Prompt template ready.")
print(f"Prompt length: {len(prompt)} chars")

Prompt template ready.
Prompt length: 129 chars


In [ ]:
import os
import re
import time
import threading
import torch
from PIL import Image

# ── CONFIG ────────────────────────────────────────────────────────────────────
batch_size     = 5
max_new_tokens = 8192
PDF_DPI        = 200   # match this in your pdf_to_images call

# ── COMPILE MODEL (one-time, ~60s) ───────────────────────────────────────────
print("Compiling model (one-time, ~60s)...")
granite_doc_model = torch.compile(granite_doc_model, mode="reduce-overhead")
print("Compiled.")

# ── RESUME FROM DISK ──────────────────────────────────────────────────────────
all_doc_tags = []
if os.path.exists(DOCTAGS_FILE):
    with open(DOCTAGS_FILE, "r", encoding="utf-8") as f:
        content = f.read()
    all_doc_tags = [t for t in content.split("\n<<PAGE_BREAK>>\n") if t.strip()]
    print(f"Resumed: {len(all_doc_tags)} pages already processed")
else:
    print("Starting fresh")

start_page       = len(all_doc_tags)
start_total      = time.time()
pages_since_save = 0

# Append-only file handle — never rewrite the whole file
progress_file = open(DOCTAGS_FILE, "a", encoding="utf-8")

# ── PREFETCH: load next batch while GPU runs current ──────────────────────────
_prefetch_result = None
_prefetch_thread = None

def _load_images(paths):
    global _prefetch_result
    _prefetch_result = [Image.open(p).convert("RGB") for p in paths]

def prefetch_next(paths):
    global _prefetch_thread
    _prefetch_thread = threading.Thread(target=_load_images, args=(paths,), daemon=True)
    _prefetch_thread.start()

def get_prefetched():
    if _prefetch_thread is not None:
        _prefetch_thread.join()
    return _prefetch_result

# ── MAIN LOOP ─────────────────────────────────────────────────────────────────
for i in range(start_page, len(image_paths), batch_size):

    batch_start  = time.time()
    batch_paths  = image_paths[i : i + batch_size]
    actual_batch = len(batch_paths)

    # First iteration loads sync; all subsequent iterations use prefetched images
    if i == start_page:
        batch_images = [Image.open(p).convert("RGB") for p in batch_paths]
    else:
        batch_images = get_prefetched()

    # Start prefetching the NEXT batch immediately (runs while GPU works below)
    next_start = i + batch_size
    if next_start < len(image_paths):
        prefetch_next(image_paths[next_start : next_start + batch_size])

    # ── ENCODE + GENERATE (batch together for max throughput) ─────────────────
    # Encoding all images in one call parallelises the vision encoder across the
    # whole batch.  Generation is then done in a single batched forward pass,
    # which is faster than one-page-at-a-time when pages are similar in length
    # (typical for resumes).  If you regularly hit OOM here, fall back to the
    # per-page loop from the alternate version below.
    inputs = granite_doc_processor(
        text          = [prompt] * actual_batch,
        images        = batch_images,
        return_tensors= "pt",
        padding       = True,          # required for batched generation
    ).to(device)

    with torch.inference_mode():
        generated_ids = granite_doc_model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            use_cache      = True,
        )

    prompt_length        = inputs.input_ids.shape[1]
    trimmed_generated_ids = generated_ids[:, prompt_length:]

    raw_outputs = granite_doc_processor.batch_decode(
        trimmed_generated_ids,
        skip_special_tokens=False,
    )
    outputs = [o.lstrip() for o in raw_outputs]

    # ── VALIDATION: fix merges and detect truncations ─────────────────────────
    validated = []
    for page_offset, raw in enumerate(outputs):
        page_num    = i + page_offset
        close_count = raw.count("</doctag>")

        if close_count == 0:
            meta = metadata[page_num] if page_num < len(metadata) else {}
            print(f"  WARNING page {page_num+1} ({meta.get('pdf_name', '')}): truncated output (no </doctag>)")
            validated.append(raw)

        elif close_count == 1:
            validated.append(raw)

        else:
            print(f"  WARNING page {page_num+1}: {close_count} pages merged — splitting")
            individual = re.findall(r"<doctag>.*?</doctag>", raw, re.DOTALL)
            if individual:
                validated.extend(individual)
                print(f"    → Split into {len(individual)} pages successfully")
            else:
                print(f"    → Split failed, keeping as-is")
                validated.append(raw)

    # ── ALIGN validated count to actual_batch ─────────────────────────────────
    if len(validated) > actual_batch:
        print(f"  NOTE: {len(validated)} outputs for {actual_batch} images — trimming to {actual_batch}")
        validated = validated[:actual_batch]
    elif len(validated) < actual_batch:
        missing = actual_batch - len(validated)
        print(f"  NOTE: {len(validated)} outputs for {actual_batch} images — padding {missing} empty entries")
        validated.extend(["<doctag></doctag>"] * missing)

    all_doc_tags.extend(validated)
    pages_since_save += actual_batch

    # ── APPEND ONLY — never rewrite the whole file ────────────────────────────
    for tag in validated:
        progress_file.write(tag + "\n<<PAGE_BREAK>>\n")
    progress_file.flush()

    # ── FREE BATCH MEMORY ─────────────────────────────────────────────────────
    # PyTorch's caching allocator handles CUDA memory reuse automatically.
    # Only call empty_cache() if you actually get an OOM — it forces a full
    # CUDA sync and costs ~100-200ms per call.
    del batch_images, inputs, generated_ids, trimmed_generated_ids

    # ── PROGRESS ──────────────────────────────────────────────────────────────
    elapsed     = time.time() - start_total
    batch_time  = time.time() - batch_start
    meta_entry  = metadata[i] if i < len(metadata) else {}
    pages_done  = len(all_doc_tags) - start_page
    s_per_page  = elapsed / max(pages_done, 1)
    eta_min     = (len(image_paths) - len(all_doc_tags)) * s_per_page / 60

    print(
        f"[{len(all_doc_tags):>5}/{len(image_paths)}] "
        f"{meta_entry.get('category', '')[:28]:28s} | "
        f"Batch: {batch_time:.1f}s | "
        f"{batch_time/actual_batch:.1f}s/page | "
        f"ETA {eta_min:.0f}min"
    )

progress_file.close()

total_elapsed = time.time() - start_total
pages_done    = len(all_doc_tags) - start_page
print(
    f"\nDone. {len(all_doc_tags)} pages total | "
    f"{total_elapsed/60:.1f}min | "
    f"{total_elapsed / max(pages_done, 1):.1f}s/page avg"
)

Compiling model (one-time, ~60s)...
Compiled.
Resumed: 20 pages already processed
[   25/5172] Accountant resumes           | Batch: 90.5s | 18.1s/page | ETA 1553min
[   30/5172] Accountant resumes           | Batch: 63.3s | 12.7s/page | ETA 1318min
[   35/5172] Accountant resumes           | Batch: 131.3s | 26.3s/page | ETA 1627min
[   40/5172] Accountant resumes           | Batch: 222.4s | 44.5s/page | ETA 2171min
[   45/5172] Accountant resumes           | Batch: 222.5s | 44.5s/page | ETA 2495min


In [ ]:
#Verify Output
with open(DOCTAGS_FILE, "r", encoding="utf-8") as f:
    content = f.read()

pages = [p for p in content.split("\n<<PAGE_BREAK>>\n") if p.strip()]
print(f"Pages in file : {len(pages)}")
print(f"Pages in RAM  : {len(all_doc_tags)}")
print(f"Pages expected: {len(image_paths)}")

# Check for truncations
truncated = [i for i, p in enumerate(pages) if "</doctag>" not in p]
if truncated:
    print(f"\nWARNING: {len(truncated)} truncated pages: {truncated[:10]}")
    print("  → Consider increasing max_new_tokens in Cell 6 and re-running")
else:
    print("\nAll pages have complete </doctag> closing tags. ✓")

# Preview first page
print(f"\nFirst page preview (first 500 chars):")
print(pages[0][:500] if pages else "(empty)")

In [ ]:
#Unload Granite
import gc
try:
    del granite_doc_model
    del granite_doc_processor
    torch.cuda.empty_cache()
    gc.collect()
    print("Granite unloaded — VRAM freed")
except NameError:
    torch.cuda.empty_cache()
    print("Granite was not in session")